# 05_train_lgbm — LightGBM Ranker/Classifierの学習と評価(5ソース×学習4週 版)

`04_feature_engineering.ipynb` で作った候補×特徴量テーブル(5候補ソース、学習4週+検証1週)でLightGBMを学習し、
LambdaRank(LGBMRanker)とbinary分類(LGBMClassifier)を比較する。

評価はvalセット(cutoff = 最終日-7日)に対する**真の実購入集合**(候補プールに入っていない購入も含む)でのMAP@12。
比較対象:
- 02_baseline.ipynb のハイブリッドルールベース: バックテストMAP@12=0.0248, Public LB=0.02000
- 前回の05(2週学習・4ソース)のLGBMRanker: バックテストMAP@12=0.0322, Public LB=0.02527
- 03の5ソース単純結合プールのRecall上限: 0.088(4ソース時点の0.082から改善)

注記: 学習のiteration数はvalセットでのearly stoppingで決めている(valはハイパーパラメータ選択にも使っており、完全に独立した評価ではない点に留意)。

In [ ]:
import os
import sys
import glob
import pickle
from datetime import datetime

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import lightgbm as lgb

sys.path.append(os.path.abspath("."))
from common import backtest_split, get_con, mapk, append_experiment_log

FEATURES_DIR = os.path.join("..", "02_output", "features")
MODELS_DIR = os.path.join("..", "02_output", "models")

train_path = sorted(glob.glob(os.path.join(FEATURES_DIR, "train_features_*.parquet")))[-1]
val_path = sorted(glob.glob(os.path.join(FEATURES_DIR, "val_features_*.parquet")))[-1]
print(f"train: {train_path}")
print(f"val:   {val_path}")

CAT_COLS = ["product_group_name", "index_group_name", "section_name"]

## 読み込み+ダウンキャスト+負例サンプリング(OOM対策)

`pd.read_parquet`でtrain(80M行, 1.8GB)とvalを両方読み込んでからダウンキャストすると、
素朴なint64/float64のままの2つの巨大DataFrameが同時にメモリ上に存在する瞬間ができてしまい、
メモリ使用量が約30GBに達してOOM killされることを確認した(jetsamログでPythonプロセスのkillを確認)。
そこで**trainを読み込み→ダウンキャスト→負例サンプリングまで完全に終わらせてから、
val用の生DataFrameを読み込む**という順序に変更し、2つの巨大な生DataFrameが同時に存在しないようにする。

ダウンキャストの内容:
1. pyarrowの`categories`引数でカテゴリ列(product_group_name/index_group_name/section_name)を
   dictionary encodingのまま`category`dtypeにする(pandasの素朴な読み込みだと行ごとにPython文字列
   オブジェクトが展開されて非常に重い)
2. `product_code`(未使用列)は読み込み時点で除外する
3. 0/1のソースフラグ・label・n_sourcesはint8、customer_idx/article_id/カウント系はint32、
   価格・比率系はfloat32、`cutoff`もcategoryへダウンキャストする

負例サンプリング(陽性率0.08%と極端に不均衡なため):
- 正解を1件でも含むグループの行は**全て残す**(ランキング学習に必要な文脈を壊さないため、グループ単位で残す/落とすを決める。行単位で間引くとgroup引数と対応が崩れる)
- 正解を1件も含まないグループ(候補プールが実購入を1つも捉えられなかった顧客)は、グループ単位でランダムに`NEG_GROUP_SAMPLE_FRAC`の割合だけ残す
- valセットは評価用(early stoppingの参照値・最終MAP@12算出)なのでそのまま使う(サイズも441MBとtrainよりずっと小さく、影響も出にくい)

In [ ]:
import gc

load_columns = [c for c in pq.ParquetFile(train_path).schema.names if c != "product_code"]

int8_cols = ["from_repeat", "from_trend", "from_category_color", "from_cf", "from_cooccurrence",
             "from_content_similarity", "n_sources", "label"]
int32_cols = ["customer_idx", "article_id", "u_total_purchases", "u_distinct_articles",
              "u_distinct_product_codes", "u_active_weeks", "i_total_sales",
              "i_days_since_last_sale", "i_sales_7d", "i_sales_28d", "ui_times_bought",
              "ui_times_bought_product_code", "target_week_start_dom"]
float32_cols = ["u_avg_price", "u_price_cv", "u_online_ratio", "u_menswear_share",
                "u_babychildren_share", "u_ladieswear_share", "i_avg_price", "ui_price_diff",
                "u_days_since_last_purchase", "ui_days_since_last_bought",
                "ui_days_since_last_bought_product_code"]


def downcast(df):
    for c in int8_cols:
        df[c] = df[c].astype("int8")
    for c in int32_cols:
        df[c] = df[c].astype("int32")
    for c in float32_cols:
        df[c] = df[c].astype("float32")
    df["cutoff"] = df["cutoff"].astype("category")
    return df


NEG_GROUP_SAMPLE_FRAC = 0.15

# --- train: 読み込み→ダウンキャスト→負例サンプリングまで完結させる ---
train_df = pq.read_table(train_path, columns=load_columns).to_pandas(categories=CAT_COLS)
print(f"train 読み込み直後: {train_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
train_df = downcast(train_df)
print(f"train ダウンキャスト後: {train_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

group_has_positive = train_df.groupby(["cutoff", "customer_idx"], observed=True)["label"].transform("max")
positive_rows = train_df[group_has_positive == 1]
negative_only_rows = train_df[group_has_positive == 0]
print(f"正解を含むグループの行数: {len(positive_rows):,}")
print(f"正解を含まないグループの行数: {len(negative_only_rows):,}")

neg_group_keys = negative_only_rows[["cutoff", "customer_idx"]].drop_duplicates()
sampled_neg_keys = neg_group_keys.sample(frac=NEG_GROUP_SAMPLE_FRAC, random_state=42)
sampled_negative_rows = negative_only_rows.merge(sampled_neg_keys, on=["cutoff", "customer_idx"])

train_df = pd.concat([positive_rows, sampled_negative_rows], ignore_index=True)
del positive_rows, negative_only_rows, sampled_negative_rows, group_has_positive
gc.collect()
print(f"サンプリング後のtrain_df: {train_df.shape}, memory: {train_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# --- val: trainの生データが既に片付いた後に読み込む ---
val_df = pq.read_table(val_path, columns=load_columns).to_pandas(categories=CAT_COLS)
print(f"val 読み込み直後: {val_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
val_df = downcast(val_df)
print(f"val ダウンキャスト後: {val_df.shape}, memory: {val_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

## 特徴量の確定

- カテゴリ列はvalの側をtrainのカテゴリ集合に合わせる(pyarrowはファイルごとに辞書順が独立なため)
- customer_idx/article_id/label/cutoffは特徴量から除外(product_codeは読み込み時点で除外済み)

In [ ]:
cat_cols = CAT_COLS
exclude_cols = ["customer_idx", "article_id", "label", "cutoff"]
feature_cols = [c for c in train_df.columns if c not in exclude_cols]

train_df = train_df.sort_values(["cutoff", "customer_idx"]).reset_index(drop=True)
val_df = val_df.sort_values(["customer_idx"]).reset_index(drop=True)

for c in cat_cols:
    train_df[c] = train_df[c].astype("category")
    val_df[c] = pd.Categorical(val_df[c], categories=train_df[c].cat.categories)

print(f"最終train_dfメモリ: {train_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

X_train, y_train = train_df[feature_cols], train_df["label"]
X_val, y_val = val_df[feature_cols], val_df["label"]

train_group_sizes = train_df.groupby(["cutoff", "customer_idx"], sort=False, observed=True).size().to_numpy()
val_group_sizes = val_df.groupby("customer_idx", sort=False).size().to_numpy()

assert train_group_sizes.sum() == len(train_df)
assert val_group_sizes.sum() == len(val_df)
print(f"feature_cols ({len(feature_cols)}): {feature_cols}")
print(f"train groups: {len(train_group_sizes):,}, val groups: {len(val_group_sizes):,}")

## LGBMRanker(LambdaRank)

In [ ]:
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    n_estimators=300,
    num_leaves=63,
    learning_rate=0.05,
    random_state=42,
    verbosity=-1,
)
ranker.fit(
    X_train, y_train,
    group=train_group_sizes,
    eval_set=[(X_val, y_val)],
    eval_group=[val_group_sizes],
    eval_at=[12],
    categorical_feature=cat_cols,
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

## LGBMClassifier(binary)

正例率が約0.08〜0.09%と極端に不均衡なため `is_unbalance=True` を指定する。

In [ ]:
classifier = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=300,
    num_leaves=63,
    learning_rate=0.05,
    is_unbalance=True,
    random_state=42,
    verbosity=-1,
)
classifier.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="binary_logloss",
    categorical_feature=cat_cols,
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

## 評価: valセットの真の実購入集合に対するMAP@12

候補プールに入っていない実購入も含めた完全な正解集合(`common.backtest_split`、02_baseline/03と同じ定義)で評価する。
候補プール自体のRecall上限(03の5ソース単純結合で0.088)を超えることはできない。

In [ ]:
con = get_con()
cutoff_check, actual_map = backtest_split(con, holdout_days=7)
print(f"cutoff_check={cutoff_check} (val_dfのcutoffと一致するはず: {val_df['cutoff'].iloc[0]})")
print(f"実購入があった顧客数: {len(actual_map):,}")

In [ ]:
def build_pred_map_from_scores(df, scores, k=12):
    tmp = df[["customer_idx", "article_id"]].copy()
    tmp["score"] = scores
    top = tmp.sort_values("score", ascending=False).groupby("customer_idx").head(k)
    return top.groupby("customer_idx")["article_id"].apply(list).to_dict()


val_df["score_ranker"] = ranker.predict(X_val)
val_df["score_classifier"] = classifier.predict_proba(X_val)[:, 1]

pred_map_ranker = build_pred_map_from_scores(val_df, val_df["score_ranker"])
pred_map_classifier = build_pred_map_from_scores(val_df, val_df["score_classifier"])

map_ranker = mapk(actual_map, pred_map_ranker, k=12)
map_classifier = mapk(actual_map, pred_map_classifier, k=12)

results_df = pd.DataFrame([
    {"model": "LGBMRanker(LambdaRank, 5ソース×学習4週)", "map@12": map_ranker},
    {"model": "LGBMClassifier(binary, 5ソース×学習4週)", "map@12": map_classifier},
    {"model": "参考: 前回05(2週学習・4ソース)LGBMRanker", "map@12": 0.0322},
    {"model": "参考: 02_baselineハイブリッド", "map@12": 0.0248},
])
results_df

## 特徴量重要度

In [ ]:
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "ranker_gain": ranker.booster_.feature_importance(importance_type="gain"),
    "classifier_gain": classifier.booster_.feature_importance(importance_type="gain"),
}).sort_values("ranker_gain", ascending=False)
importance_df

## モデル保存と実験ログ記録

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
best_name = "ranker" if map_ranker >= map_classifier else "classifier"
best_model = ranker if best_name == "ranker" else classifier
best_score = max(map_ranker, map_classifier)

ranker_path = os.path.join(MODELS_DIR, f"model_{timestamp}_ranker.pkl")
classifier_path = os.path.join(MODELS_DIR, f"model_{timestamp}_classifier.pkl")
with open(ranker_path, "wb") as f:
    pickle.dump({"model": ranker, "feature_cols": feature_cols, "cat_cols": cat_cols}, f)
with open(classifier_path, "wb") as f:
    pickle.dump({"model": classifier, "feature_cols": feature_cols, "cat_cols": cat_cols}, f)
print(f"保存しました: {ranker_path}")
print(f"保存しました: {classifier_path}")

params_desc = f"n_estimators=300(early_stopping=30), num_leaves=63, learning_rate=0.05, neg_group_sample_frac={NEG_GROUP_SAMPLE_FRAC}"
features_desc = ", ".join(feature_cols)

append_experiment_log(
    run_id=f"run_{timestamp}_ranker", timestamp=timestamp, notebook="05_train_lgbm.ipynb",
    description="LGBMRanker(LambdaRank, group=customer_idx x cutoff)。04の候補×特徴量テーブル(5ソース×学習4週)で学習",
    features=features_desc, params=f"objective=lambdarank, {params_desc}",
    cv_score=round(map_ranker, 4), public_lb="", submission_file="", model_file=os.path.basename(ranker_path),
    notes="valは5候補ソース結合プール由来(学習は4週分)。評価は候補プール外の実購入も含む完全な正解集合に対するMAP@12",
)
append_experiment_log(
    run_id=f"run_{timestamp}_classifier", timestamp=timestamp, notebook="05_train_lgbm.ipynb",
    description="LGBMClassifier(binary, is_unbalance=True)。04の候補×特徴量テーブル(5ソース×学習4週)で学習",
    features=features_desc, params=f"objective=binary, is_unbalance=True, {params_desc}",
    cv_score=round(map_classifier, 4), public_lb="", submission_file="", model_file=os.path.basename(classifier_path),
    notes="valは5候補ソース結合プール由来(学習は4週分)。評価は候補プール外の実購入も含む完全な正解集合に対するMAP@12",
).tail(2)

## 次のステップ

- MAP@12が02_baselineを上回っていれば、cutoff=最終日(全データ)で候補生成→特徴量→予測を行い、
  全顧客(コールドスタートは人気度フォールバック)向けの提出ファイルを作って06として提出する
- 下回っていれば、特徴量重要度を見て弱いソース/特徴量を見直す、学習週を増やす、ハイパーパラメータを調整するなどの改善を検討する